# LightFM User Split (80/20) with TinyBERT + KGE Features (From Preprocessed Outputs)
Notebook nay import truc tiep outputs da xu ly tu 2 pipeline TinyBERT va KGE, ket hop features, sau do train/evaluate LightFM voi cau hinh baseline de so sanh cong bang.

## 1. Environment Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

np.random.seed(42)

### Optional one-time dependency installation
Run only if your current kernel is missing required packages.

In [2]:
pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 24.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=825357 sha256=6c121714bd48eae7ad5f089219bc98b5cf765d8a53acb9c506b8050bb9970a00
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


## 2. Resolve TinyBERT and KGE Artifact Paths

In [3]:
TB_DIR = Path(".")
KGE_DIR = Path(".")
HYBRID_OUT_DIR = Path(".")
HYBRID_OUT_DIR.mkdir(parents=True, exist_ok=True)

TB_FILES = {
    "movies_clean": TB_DIR / "01clean_movies_clean.csv",
    "users_clean": TB_DIR / "01clean_users_clean.csv",
    "all_item_ids": TB_DIR / "02split_all_item_ids.npy",
    "train_users": TB_DIR / "02split_train_users.csv",
    "test_users": TB_DIR / "02split_test_users.csv",
    "tinybert_item_ids": TB_DIR / "03tinybert_tinybert_item_ids.npy",
    "tinybert_vectors": TB_DIR / "03tinybert_tinybert_vectors.npy",
}

KGE_FILES = {
    "movies_clean": KGE_DIR / "01clean_movies.csv",
    "users_clean": KGE_DIR / "01clean_users.csv",
    "movie_content": KGE_DIR / "02movie_content_features.csv",
    "user_behavior": KGE_DIR / "02user_behavior_features.csv",
    "train_ratings": KGE_DIR / "03split_train_ratings.csv",
    "test_ratings": KGE_DIR / "03split_test_ratings.csv",
    "all_item_ids": KGE_DIR / "03split_all_item_ids.npy",
    "kge_item_embeddings": KGE_DIR / "05kge_item_embeddings.csv",
    "kge_user_embeddings": KGE_DIR / "05kge_user_embeddings.csv",
}

required_files = {**TB_FILES, **KGE_FILES}
missing = [str(path) for path in required_files.values() if not Path(path).exists()]
if missing:
    preview = "\n".join(missing[:20])
    raise FileNotFoundError(f"Missing required artifacts:\n{preview}")

print(f"TinyBERT artifacts: {TB_DIR.resolve()}")
print(f"KGE artifacts: {KGE_DIR.resolve()}")
print(f"Hybrid output: {HYBRID_OUT_DIR.resolve()}")

TinyBERT artifacts: /content
KGE artifacts: /content
Hybrid output: /content


## 3. Load Artifacts and Validate Split Consistency

In [4]:
movies_tb = pd.read_csv(TB_FILES["movies_clean"]).copy()
users_tb = pd.read_csv(TB_FILES["users_clean"]).copy()

movies_kge = pd.read_csv(KGE_FILES["movies_clean"]).copy()
users_kge = pd.read_csv(KGE_FILES["users_clean"]).copy()
movie_content = pd.read_csv(KGE_FILES["movie_content"]).copy()
user_behavior = pd.read_csv(KGE_FILES["user_behavior"]).copy()
train_df = pd.read_csv(KGE_FILES["train_ratings"]).copy()
test_df = pd.read_csv(KGE_FILES["test_ratings"]).copy()

tb_train_users = set(pd.read_csv(TB_FILES["train_users"])["user_id"].astype(int).tolist())
tb_test_users = set(pd.read_csv(TB_FILES["test_users"])["user_id"].astype(int).tolist())

tb_item_ids = np.load(TB_FILES["tinybert_item_ids"]).astype(np.int64)
tb_vectors = np.load(TB_FILES["tinybert_vectors"]).astype(np.float32)
if len(tb_item_ids) != len(tb_vectors):
    raise ValueError("TinyBERT item_ids and vectors length mismatch.")
tinybert_lookup = {int(item_id): vec for item_id, vec in zip(tb_item_ids.tolist(), tb_vectors)}

all_item_ids_tb = set(np.load(TB_FILES["all_item_ids"]).astype(np.int64).tolist())
all_item_ids_kge = set(np.load(KGE_FILES["all_item_ids"]).astype(np.int64).tolist())
all_item_ids = sorted(int(x) for x in (all_item_ids_tb | all_item_ids_kge))
all_item_set = set(all_item_ids)

for col in ["user_id", "movie_id", "rating"]:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
train_df = train_df.dropna(subset=["user_id", "movie_id", "rating"]).copy()
test_df = test_df.dropna(subset=["user_id", "movie_id", "rating"]).copy()
train_df["user_id"] = train_df["user_id"].astype(int)
train_df["movie_id"] = train_df["movie_id"].astype(int)
test_df["user_id"] = test_df["user_id"].astype(int)
test_df["movie_id"] = test_df["movie_id"].astype(int)

train_df = train_df[train_df["movie_id"].isin(all_item_set)].copy()
test_df = test_df[test_df["movie_id"].isin(all_item_set)].copy()

kge_train_users = set(train_df["user_id"].astype(int).unique().tolist())
kge_test_users = set(test_df["user_id"].astype(int).unique().tolist())
all_user_ids = sorted(kge_train_users | kge_test_users)
all_user_set = set(all_user_ids)

kge_item_df = pd.read_csv(KGE_FILES["kge_item_embeddings"]).copy()
kge_user_df = pd.read_csv(KGE_FILES["kge_user_embeddings"]).copy()

kge_item_id_col = "item_id" if "item_id" in kge_item_df.columns else ("movie_id" if "movie_id" in kge_item_df.columns else None)
if kge_item_id_col is None:
    raise ValueError("KGE item embedding file must contain item_id or movie_id.")
if "user_id" not in kge_user_df.columns:
    raise ValueError("KGE user embedding file must contain user_id.")

kge_item_cols = [c for c in kge_item_df.columns if c.startswith("kge_")]
kge_user_cols = [c for c in kge_user_df.columns if c.startswith("kge_")]
if not kge_item_cols or not kge_user_cols:
    raise ValueError("Expected KGE columns prefixed by 'kge_'.")

kge_item_lookup = {
    int(row[kge_item_id_col]): row[kge_item_cols].to_numpy(dtype=np.float32)
    for _, row in kge_item_df.iterrows()
}
kge_user_lookup = {
    int(row["user_id"]): row[kge_user_cols].to_numpy(dtype=np.float32)
    for _, row in kge_user_df.iterrows()
}

movies_tb["id"] = pd.to_numeric(movies_tb.get("id"), errors="coerce")
movies_tb = movies_tb.dropna(subset=["id"]).copy()
movies_tb["id"] = movies_tb["id"].astype(int)
movies_tb = movies_tb.drop_duplicates(subset=["id"]).copy()

movie_content["movie_id"] = pd.to_numeric(movie_content.get("movie_id"), errors="coerce")
movie_content = movie_content.dropna(subset=["movie_id"]).copy()
movie_content["movie_id"] = movie_content["movie_id"].astype(int)
movie_content = movie_content.drop_duplicates(subset=["movie_id"]).copy()

user_behavior["user_id"] = pd.to_numeric(user_behavior.get("user_id"), errors="coerce")
user_behavior = user_behavior.dropna(subset=["user_id"]).copy()
user_behavior["user_id"] = user_behavior["user_id"].astype(int)
user_behavior = user_behavior.drop_duplicates(subset=["user_id"]).copy()

if tb_train_users != kge_train_users or tb_test_users != kge_test_users:
    print("WARNING: TinyBERT and KGE split users are not identical. This notebook uses KGE train/test ratings as anchor split.")

tinybert_item_cov = len(set(tinybert_lookup.keys()) & all_item_set) / max(1, len(all_item_set))
kge_item_cov = len(set(kge_item_lookup.keys()) & all_item_set) / max(1, len(all_item_set))
kge_user_cov = len(set(kge_user_lookup.keys()) & all_user_set) / max(1, len(all_user_set))

print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Users (train/test): {len(kge_train_users):,}/{len(kge_test_users):,} | Total mapped users: {len(all_user_ids):,}")
print(f"Items mapped: {len(all_item_ids):,}")
print(f"TinyBERT item coverage: {tinybert_item_cov:.3f}")
print(f"KGE item coverage: {kge_item_cov:.3f}")
print(f"KGE user coverage: {kge_user_cov:.3f}")
print(f"TinyBERT vectors shape: {tb_vectors.shape}")
print(f"KGE dims (item/user): {len(kge_item_cols)}/{len(kge_user_cols)}")

Train rows: 938,842 | Test rows: 233,192
Users (train/test): 489/122 | Total mapped users: 611
Items mapped: 78,628
TinyBERT item coverage: 1.000
KGE item coverage: 1.000
KGE user coverage: 1.000
TinyBERT vectors shape: (78628, 312)
KGE dims (item/user): 64/64


## 4. Ranking Evaluation Utilities

In [5]:
def sample_negatives(candidate_pool, exclude_set, n_negatives=20, rng=None):
    rng = rng or np.random.default_rng(42)
    pool = np.array([i for i in candidate_pool if i not in exclude_set], dtype=int)
    if len(pool) == 0:
        return np.array([], dtype=int)
    replace = len(pool) < n_negatives
    return rng.choice(pool, size=n_negatives, replace=replace)

def ranking_metrics_from_rank(rank, k=10):
    recall = 1.0 if rank <= k else 0.0
    ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0.0
    mrr = 1.0 / rank
    return recall, ndcg, mrr

def rank_positive_among_candidates(scores, positive_mask, rng=None):
    rng = rng or np.random.default_rng(42)
    positive_idx = int(np.flatnonzero(positive_mask)[0])
    pos_score = float(scores[positive_idx])

    better = int(np.sum(scores > pos_score))
    tie_indices = np.flatnonzero(scores == pos_score)
    tie_count = int(len(tie_indices))

    if tie_count <= 1:
        return better + 1, tie_count

    shuffled_ties = tie_indices[rng.permutation(tie_count)]
    tie_pos = int(np.where(shuffled_ties == positive_idx)[0][0])
    return better + tie_pos + 1, tie_count

def evaluate_ranker(score_fn, positives_df, candidate_pool, all_user_pos_lookup, n_negatives=20, k=10, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    tie_cases = 0

    for uid, group in positives_df.groupby("user_id"):
        uid = int(uid)
        known_positives = all_user_pos_lookup.get(uid, set())

        for _, row in group.iterrows():
            pos_item = int(row["movie_id"])
            exclude = set(known_positives)
            exclude.add(pos_item)

            negatives = sample_negatives(candidate_pool, exclude, n_negatives=n_negatives, rng=rng)
            candidate_items = np.array([pos_item] + negatives.tolist(), dtype=int)
            positive_mask = np.zeros(len(candidate_items), dtype=bool)
            positive_mask[0] = True

            perm = rng.permutation(len(candidate_items))
            candidate_items = candidate_items[perm]
            positive_mask = positive_mask[perm]

            scores = score_fn(uid, candidate_items)
            rank, tie_count = rank_positive_among_candidates(scores, positive_mask, rng=rng)
            if tie_count > 1:
                tie_cases += 1

            recall, ndcg, mrr = ranking_metrics_from_rank(rank, k=k)
            rows.append((recall, ndcg, mrr))

    if not rows:
        return {"Recall@10": np.nan, "NDCG@10": np.nan, "MRR": np.nan, "n_eval": 0, "tie_rate": np.nan}

    arr = np.array(rows, dtype=float)
    n_eval = int(len(rows))
    return {
        "Recall@10": float(arr[:, 0].mean()),
        "NDCG@10": float(arr[:, 1].mean()),
        "MRR": float(arr[:, 2].mean()),
        "n_eval": n_eval,
        "tie_rate": float(tie_cases / n_eval),
    }

## 5. Build LightFM Dataset with TinyBERT + KGE Hybrid Features

In [6]:
from lightfm.data import Dataset

MAX_TOPICS_PER_MOVIE = 5

def split_tokens(value):
    if pd.isna(value) or value == "":
        return []
    return [t.strip() for t in str(value).split("|") if t.strip()]

def sanitize_token(token):
    token = str(token).strip().lower()
    token = re.sub(r"\s+", "_", token)
    token = re.sub(r"[^a-z0-9_]+", "", token)
    return token or "unknown"

def to_float(value, default=0.0):
    numeric = pd.to_numeric(value, errors="coerce")
    return default if pd.isna(numeric) else float(numeric)

def build_user_pos_lookup(df):
    lookup = {}
    for uid, group in df.groupby("user_id"):
        lookup[int(uid)] = set(group["movie_id"].astype(int).tolist())
    return lookup

def merge_user_pos_lookups(*lookups):
    merged = {}
    for lookup in lookups:
        for uid, items in lookup.items():
            if uid not in merged:
                merged[uid] = set()
            merged[uid].update(items)
    return merged

def prepare_interactions(df):
    return [(int(r["user_id"]), int(r["movie_id"])) for _, r in df.iterrows()]

for col in ["genres", "topics", "actors", "directors", "plot", "main_title"]:
    if col not in movies_tb.columns:
        movies_tb[col] = ""
movies_tb["year_published"] = pd.to_numeric(movies_tb.get("year_published"), errors="coerce")

movie_content_map = {
    int(row.movie_id): row._asdict()
    for row in movie_content.itertuples(index=False)
}

movie_tb_map = {
    int(row.id): row._asdict()
    for row in movies_tb.itertuples(index=False)
}

user_behavior_map = {
    int(row.user_id): row._asdict()
    for row in user_behavior.itertuples(index=False)
}
behavior_cols = [c for c in user_behavior.columns if c not in {"user_id", "log_followers"}]

def build_item_feature_dicts(item_ids):
    rows = []
    for item_id in item_ids:
        item_id = int(item_id)
        content = movie_content_map.get(item_id, {})
        meta = movie_tb_map.get(item_id, {})
        feat = {}

        for g in split_tokens(content.get("genres", meta.get("genres", ""))):
            feat[f"genre__{sanitize_token(g)}"] = 1.0

        topics = split_tokens(content.get("topics", meta.get("topics", "")))[:MAX_TOPICS_PER_MOVIE]
        for t in topics:
            feat[f"topic__{sanitize_token(t)}"] = 1.0

        for actor in split_tokens(meta.get("actors", "")):
            feat[f"actor__{sanitize_token(actor)}"] = 1.0

        for director in split_tokens(meta.get("directors", "")):
            feat[f"director__{sanitize_token(director)}"] = 1.0

        country_code = str(content.get("country_code", "")).strip().upper()
        if country_code and country_code != "NAN":
            feat[f"country__{sanitize_token(country_code)}"] = 1.0

        feat["year_norm"] = to_float(content.get("year_norm", 0.0), default=0.0)
        if "has_plot" in content:
            feat["has_plot"] = to_float(content.get("has_plot", 0.0), default=0.0)
        else:
            has_plot = str(meta.get("plot", "")).strip() != ""
            feat["has_plot"] = float(has_plot)

        tb_vec = tinybert_lookup.get(item_id)
        if tb_vec is not None:
            for idx, value in enumerate(tb_vec):
                feat[f"text_emb__{idx:03d}"] = float(value)

        kge_vec = kge_item_lookup.get(item_id)
        if kge_vec is not None:
            for idx, value in enumerate(kge_vec):
                feat[f"kge_item__{idx:03d}"] = float(value)

        rows.append((item_id, feat))

    return rows

def build_user_feature_dicts(user_ids):
    rows = []
    for uid in user_ids:
        uid = int(uid)
        behavior = user_behavior_map.get(uid, {})
        feat = {}

        for col in behavior_cols:
            raw_value = behavior.get(col, 0.0)
            feat[f"behavior__{sanitize_token(col)}"] = to_float(raw_value, default=0.0)

        feat["log_followers"] = to_float(behavior.get("log_followers", 0.0), default=0.0)

        kge_vec = kge_user_lookup.get(uid)
        if kge_vec is not None:
            for idx, value in enumerate(kge_vec):
                feat[f"kge_user__{idx:03d}"] = float(value)

        rows.append((uid, feat))

    return rows

train_df_lfm = train_df[
    train_df["user_id"].isin(all_user_set)
    & train_df["movie_id"].isin(all_item_set)
].copy()
test_df_eval = test_df[
    test_df["user_id"].isin(all_user_set)
    & test_df["movie_id"].isin(all_item_set)
].copy()

train_user_pos = build_user_pos_lookup(train_df_lfm)
test_user_pos = build_user_pos_lookup(test_df_eval)
all_user_pos = merge_user_pos_lookups(train_user_pos, test_user_pos)

item_features = build_item_feature_dicts(all_item_ids)
user_features = build_user_feature_dicts(all_user_ids)

dataset = Dataset()
dataset.fit(
    users=all_user_ids,
    items=all_item_ids,
    user_features=sorted({f for _, d in user_features for f in d.keys()}),
    item_features=sorted({f for _, d in item_features for f in d.keys()}),
)

train_interactions, _ = dataset.build_interactions(prepare_interactions(train_df_lfm))
user_features_mx = dataset.build_user_features(user_features)
item_features_mx = dataset.build_item_features(item_features)

user_id_map, _, item_id_map, _ = dataset.mapping()

print(f"LightFM users: {len(user_id_map):,} | items: {len(item_id_map):,}")
print(f"Train interactions: {train_interactions.getnnz():,}")
print(f"User feature matrix: {user_features_mx.shape}")
print(f"Item feature matrix: {item_features_mx.shape}")

LightFM users: 611 | items: 78,628
Train interactions: 938,842
User feature matrix: (611, 682)
Item feature matrix: (78628, 343657)


## 6. Train LightFM (Baseline Configuration)

In [7]:
from lightfm import LightFM

# Keep the same baseline configuration as your LightFM notebooks.
model = LightFM(loss="warp", no_components=64, learning_rate=0.05, random_state=42)
model.fit(
    train_interactions,
    user_features=user_features_mx,
    item_features=item_features_mx,
    epochs=30,
    num_threads=1,
    verbose=True,
)

Epoch: 100%|██████████| 30/30 [2:04:49<00:00, 249.65s/it]


## 7. Evaluate on User-Split Test Set and Save Metrics

In [8]:
def hybrid_lfm_score_fn(uid, candidate_items):
    if uid not in user_id_map:
        return np.zeros(len(candidate_items), dtype=float)

    uidx = user_id_map[uid]
    item_idxs = []
    valid_positions = []

    for pos, item in enumerate(candidate_items):
        if item in item_id_map:
            item_idxs.append(item_id_map[item])
            valid_positions.append(pos)

    scores = np.zeros(len(candidate_items), dtype=float)
    if item_idxs:
        preds = model.predict(
            uidx,
            np.array(item_idxs, dtype=int),
            user_features=user_features_mx,
            item_features=item_features_mx,
        )
        for pos, sc in zip(valid_positions, preds):
            scores[pos] = float(sc)

    return scores

test_metrics = evaluate_ranker(
    hybrid_lfm_score_fn,
    positives_df=test_df_eval,
    candidate_pool=all_item_set,
    all_user_pos_lookup=all_user_pos,
    n_negatives=20,
    k=10,
    seed=43,
)

metrics_path = HYBRID_OUT_DIR / "06lightfm_tb_kge_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

run_config = {
    "loss": "warp",
    "no_components": 64,
    "learning_rate": 0.05,
    "epochs": 30,
    "num_threads": 1,
    "n_negatives": 20,
    "k": 10,
    "seed": 43,
}
config_path = HYBRID_OUT_DIR / "06lightfm_tb_kge_run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print("LightFM Hybrid (TinyBERT + KGE, User Split 80/20)")
print("Test:", test_metrics)
print(f"Saved metrics to: {metrics_path}")
print(f"Saved run config to: {config_path}")

LightFM Hybrid (TinyBERT + KGE, User Split 80/20)
Test: {'Recall@10': 0.8847258911111874, 'NDCG@10': 0.6864041753557729, 'MRR': 0.6303517715463927, 'n_eval': 233192, 'tie_rate': 3.001818244193626e-05}
Saved metrics to: 06lightfm_tb_kge_metrics.json
Saved run config to: 06lightfm_tb_kge_run_config.json


## 8. Save Deployment Bundle for Production
Save all artifacts needed for real-world inference: trained model, id mappings, feature matrices, and metadata.

In [10]:
import pickle
from datetime import datetime, timezone
import json # Ensure json is imported if not already

from scipy.sparse import save_npz
import numpy as np # Ensure numpy is imported for RandomState check

DEPLOY_DIR = HYBRID_OUT_DIR / "07deployment_bundle"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

model_path = DEPLOY_DIR / "lightfm_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)

user_id_map_deploy, user_feature_map, item_id_map_deploy, item_feature_map = dataset.mapping()

def as_jsonable_mapping(mapping):
    return {str(k): int(v) for k, v in mapping.items()}

def invert_id_map(mapping):
    inverse = {}
    for raw_id, idx in mapping.items():
        if isinstance(raw_id, (np.integer, int)):
            inverse[str(int(idx))] = int(raw_id)
        else:
            inverse[str(int(idx))] = str(raw_id)
    return inverse

mapping_payload = {
    "user_id_map": as_jsonable_mapping(user_id_map_deploy),
    "item_id_map": as_jsonable_mapping(item_id_map_deploy),
    "index_to_user_id": invert_id_map(user_id_map_deploy),
    "index_to_item_id": invert_id_map(item_id_map_deploy),
    "user_feature_map": as_jsonable_mapping(user_feature_map),
    "item_feature_map": as_jsonable_mapping(item_feature_map),
}
mapping_path = DEPLOY_DIR / "mappings.json"
with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(mapping_payload, f, indent=2, ensure_ascii=False)

user_features_path = DEPLOY_DIR / "user_features.npz"
item_features_path = DEPLOY_DIR / "item_features.npz"
save_npz(user_features_path, user_features_mx.tocsr())
save_npz(item_features_path, item_features_mx.tocsr())

all_item_ids_path = DEPLOY_DIR / "all_item_ids.npy"
np.save(all_item_ids_path, np.array(all_item_ids, dtype=np.int64))

training_config_payload = run_config if "run_config" in globals() else {}
metrics_payload = test_metrics if "test_metrics" in globals() else {}

# Get model parameters and handle non-serializable objects
model_init_params = model.get_params()
cleaned_model_init_params = {}
for key, value in model_init_params.items():
    if isinstance(value, np.random.RandomState):
        # Try to get the seed, otherwise use a string representation
        try:
            # The state of RandomState can be a tuple, often the second element (an array) has the seed
            # This might vary based on numpy version, but often state[1][0] is the seed integer
            seed_value = value.get_state()[1][0]
            if isinstance(seed_value, (np.integer, int)):
                cleaned_model_init_params[key] = int(seed_value)
            else:
                cleaned_model_init_params[key] = str(value)
        except Exception:
            cleaned_model_init_params[key] = str(value)
    else:
        cleaned_model_init_params[key] = value

metadata_payload = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_type": "LightFM",
    "model_init_params": cleaned_model_init_params, # Use the cleaned parameters
    "training_config": training_config_payload,
    "metrics": metrics_payload,
    "counts": {
        "n_users": len(user_id_map_deploy),
        "n_items": len(item_id_map_deploy),
        "n_user_features": len(user_feature_map),
        "n_item_features": len(item_feature_map),
    },
    "files": {
        "model": model_path.name,
        "mappings": mapping_path.name,
        "user_features": user_features_path.name,
        "item_features": item_features_path.name,
        "all_item_ids": all_item_ids_path.name,
    },
}
metadata_path = DEPLOY_DIR / "bundle_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_payload, f, indent=2, ensure_ascii=False)

print("Saved deployment bundle:")
print(f"- Model: {model_path}")
print(f"- Mappings: {mapping_path}")
print(f"- User features: {user_features_path}")
print(f"- Item features: {item_features_path}")
print(f"- Item catalog: {all_item_ids_path}")
print(f"- Metadata: {metadata_path}")

Saved deployment bundle:
- Model: 07deployment_bundle/lightfm_model.pkl
- Mappings: 07deployment_bundle/mappings.json
- User features: 07deployment_bundle/user_features.npz
- Item features: 07deployment_bundle/item_features.npz
- Item catalog: 07deployment_bundle/all_item_ids.npy
- Metadata: 07deployment_bundle/bundle_metadata.json
